# Pipeline Notebook

Walks through the evaluation pipeline for the **community-led GenAI evaluation study**.

- **Stage 1 (build) and Stage 5 (analysis) run here on your laptop.**
- **Stages 2-4 run on Vista** (GPU) via the scripts in `scripts/` and `jobs/run_inference.slurm`;
  those cells are documented but not executed locally.

See `templates/methodology.md` for the full plan and `framework/data-spec.md` for the schemas.

In [1]:
import subprocess, sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Resolve the project root whether the notebook runs from notebooks/ or the repo root
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
print('Project root:', ROOT)

Project root: /work/11593/kliles/vista/hpc-research-starter


## Stage 1 - Build the prompt dataset (laptop)

Runs `scripts/build_dataset.py`, which unions the authored + BBQ-seeded (+ community) prompts,
validates the schema, and writes `data/processed/prompts.csv`.

In [2]:
result = subprocess.run([sys.executable, 'scripts/build_dataset.py'],
                        cwd=ROOT, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

prompts = pd.read_csv(ROOT / 'data' / 'processed' / 'prompts.csv')
print(f'\nLoaded {len(prompts)} prompts')
prompts.head()

Assembling prompt set...
  loaded  60 rows from community-prompts.csv
  loaded   5 rows from bbq-seeded-prompts.csv
  (skip) community-contributed.csv not found

Wrote 65 prompts -> data/processed/prompts.csv

Coverage by domain:
domain
academic-advising           7
grad-school-career          7
financial-aid               6
writing-support             6
stem-coding                 6
health-wellness             6
identity-culture-history    6
local-community             6
civic-legal                 5
mental-health               5
bias-probe                  5

Validated by community: 0 / 65


Loaded 65 prompts


,prompt_id,prompt_text,domain,primary_dimension,high_stakes,source,community_validated,bbq_example_id,bbq_category,bbq_context_condition,unbiased_answer
0,P001,I'm a sophomore at an HBCU thinking about swit...,academic-advising,effectiveness,no,authored,no,NaN,NaN,NaN,NaN
1,P002,I failed a required math course and I'm scared...,academic-advising,effectiveness,no,authored,no,NaN,NaN,NaN,NaN
2,P003,My advisor is hard to reach. Can you tell me e...,academic-advising,trustworthiness,no,authored,no,NaN,NaN,NaN,NaN
3,P004,I'm a first-generation college student and I d...,academic-advising,equity,no,authored,no,NaN,NaN,NaN,NaN
4,P005,I work 25 hours a week to pay for school. How ...,academic-advising,effectiveness,no,authored,no,NaN,NaN,NaN,NaN


## Explore the dataset

Coverage by domain, primary dimension, and high-stakes flag.

In [3]:
print('By domain:')
print(prompts['domain'].value_counts().to_string())
print('\nBy dimension:')
print(prompts['primary_dimension'].value_counts().to_string())
print('\nHigh-stakes:')
print(prompts['high_stakes'].value_counts().to_string())
print('\nBy source:')
print(prompts['source'].value_counts().to_string())

By domain:
domain
academic-advising           7
grad-school-career          7
financial-aid               6
writing-support             6
stem-coding                 6
health-wellness             6
identity-culture-history    6
local-community             6
civic-legal                 5
mental-health               5
bias-probe                  5

By dimension:
primary_dimension
equity             28
trustworthiness    21
effectiveness      16

High-stakes:
high_stakes
no     38
yes    27

By source:
source
authored                                               60
BBQ Race_ethnicity (Parrish et al. 2022; CC-BY 4.0)     5


## Visualize coverage

Saves a 300-DPI figure to `figures/` (project convention).

In [4]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

prompts['domain'].value_counts().sort_values().plot.barh(
    ax=axes[0], color='#3b7dd8')
axes[0].set_title('Prompts by domain')
axes[0].set_xlabel('count')

prompts['primary_dimension'].value_counts().plot.bar(
    ax=axes[1], color=['#3b7dd8', '#e1812c', '#3a923a'])
axes[1].set_title('Prompts by primary dimension')
axes[1].set_ylabel('count')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
(ROOT / 'figures').mkdir(exist_ok=True)
fig.savefig(ROOT / 'figures' / 'dataset_coverage.png', dpi=300, bbox_inches='tight')
print('Saved figures/dataset_coverage.png')
plt.show()

Saved figures/dataset_coverage.png


## Stage 2 - Download models + BBQ (Vista)

Run on Vista, not here (needs `$SCRATCH` + a HuggingFace login):

```bash
python scripts/download_data.py
```

## Stage 3 - Run inference (Vista, GPU)

**Pilot first**, then submit the full sweep:

```bash
# pilot - one small model, 10 prompts
python scripts/run_inference.py --limit 10 --models meta-llama/Meta-Llama-3-8B-Instruct

# full sweep via Slurm
sbatch jobs/run_inference.slurm
```

Produces `data/processed/responses.csv`.

## Stage 4 - Score responses (Vista judge, then laptop)

```bash
python scripts/evaluate.py
```

Produces `results/scores.csv` (rubric scores + BBQ standard score + divergence).

## Stage 5 - Analysis preview (laptop)

Runs once `results/scores.csv` exists. Until then it prints a friendly notice instead of erroring.

In [5]:
scores_path = ROOT / 'results' / 'scores.csv'
if not scores_path.exists():
    print('No results/scores.csv yet - run Stages 2-4 on Vista first.')
else:
    scores = pd.read_csv(scores_path)
    print(f'Loaded {len(scores)} scored rows')
    # Mean community dimension scores per model
    dims = ['effectiveness', 'equity', 'trustworthiness']
    print('\nMean dimension scores by model:')
    print(scores.groupby('model_id')[dims].mean().round(2).to_string())
    # Divergence: community vs. standard
    if 'divergence' in scores.columns:
        ax = scores.groupby('model_id')['divergence'].mean().plot.bar(
            color='#3b7dd8', figsize=(8, 5))
        ax.set_title('Mean community-vs-standard divergence by model')
        ax.set_ylabel('divergence (community - standard)')
        plt.tight_layout()
        plt.savefig(ROOT / 'figures' / 'divergence_by_model.png', dpi=300, bbox_inches='tight')
        plt.show()

No results/scores.csv yet - run Stages 2-4 on Vista first.
